# 04A — SIMCA grid validation

Goal: run the main SIMCA grid search on pure validation data.

Protocol:

- calibration: pure peanut objects from batches 1 and 2;
- validation: pure almond and pure peanut objects from batch 3;
- no batch 4 is used in this notebook;
- no mixture image is used in this notebook.

This notebook produces a broad candidate panel. Robustness filtering is done later in:

`04B_simca_validation_robustness.ipynb`.

Essential outputs:

- `standard_grid_summary.parquet`
- `empirical_cv_grid_summary.parquet`
- `combined_grid_summary.parquet`
- `selected_candidate_configs.parquet`
- `grid_validation_protocol.parquet`

In [1]:
from __future__ import annotations

import sys
import json
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 300)

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Run this notebook from the project root or notebooks/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


In [2]:
from src.io.database_h5 import load_nir_uco_h5

from src.utils import (
    save_parquet,
    save_parquet_if_nonempty,
    load_parquet,
    list_result_files,
)

from src.spectra.preprocessing_configs import normalize_preprocessing_configs
from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)

from src.decision.uncertainty import (
    calibrate_three_way_thresholds_by_config,
)

from src.workflows.simca import (
    make_target_train_filters,
    run_simca_pixel_projection_grid,
    run_simca_rule_variant_grid,
    refit_selected_simca_configs,
)

from src.workflows.simca_selection_utils import (
    normalize_simca_rule_columns,
    add_detection_selection_score,
    sort_detection_selection,
    add_reference_selection_scores,
    select_top_models,
    ensure_candidate_columns,
    fill_selected_config_defaults,
    summarize_parameter_tendencies,
    sequential_pareto_filter,
)

from src.visualization.plot_diagnostics import (
    plot_metric_heatmap,
)

%load_ext autoreload
%autoreload 2

# 1. Params & config

In [3]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
DB_H5_PATH = PROJECT_ROOT / "HSI Data" / "processed" / "nir_uco_database.h5"

# ---------------------------------------------------------------------
# Spectral configuration
# ---------------------------------------------------------------------
USE_WAVELENGTH_WINDOW = False
WAVELENGTH_MODE = "non_noisy_all"

WINDOW_MIN_NM = 1225.0
WINDOW_MAX_NM = 1675.0

if USE_WAVELENGTH_WINDOW:
    RESULTS_TAG = f"{int(WINDOW_MIN_NM)}_{int(WINDOW_MAX_NM)}"
else:
    RESULTS_TAG = "non_noisy_all"

# ---------------------------------------------------------------------
# Output paths
# ---------------------------------------------------------------------
RESULTS_DIR = PROJECT_ROOT / "results" / f"04A_simca_grid_validation_{RESULTS_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

#STANDARD_GRID_SUMMARY_PATH = RESULTS_DIR / "standard_grid_summary.parquet"
#STANDARD_GRID_ERRORS_PATH = RESULTS_DIR / "standard_grid_errors.parquet"

#EMPIRICAL_GRID_SUMMARY_PATH = RESULTS_DIR / "empirical_cv_grid_summary.parquet"
#EMPIRICAL_GRID_ERRORS_PATH = RESULTS_DIR / "empirical_cv_grid_errors.parquet"

GRID_SUMMARY_PATH = RESULTS_DIR / "grid_summary.parquet"
PARETO_2WAY_PATH = RESULTS_DIR / "pareto_2way_candidates_large.parquet"
THREE_WAY_GRID_PATH = RESULTS_DIR / "three_way_threshold_grid.parquet"
SELECTED_CANDIDATE_CONFIGS_PATH = RESULTS_DIR / "selected_candidate_configs.parquet"
GRID_VALIDATION_PROTOCOL_PATH = RESULTS_DIR / "grid_validation_protocol.parquet"

PCA_SELECTED_PREPROCESSINGS_PATH = (
    PROJECT_ROOT
    / "results"
    / f"03_pca_{RESULTS_TAG}"
    / "pca_selected_preprocessings.parquet"
)



# ---------------------------------------------------------------------
# Detection protocol
# ---------------------------------------------------------------------
TARGET_CLASS = "peanut"
NON_TARGET_LABEL = "non_target"
REFERENCE_CLASSES = ("almond", TARGET_CLASS)

TRAIN_FILTERS = make_target_train_filters(
    target_class=TARGET_CLASS,
    train_batches=[1, 2],
)

VALIDATION_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": list(REFERENCE_CLASSES),
    "batch": [3],
}

# Explicitly documented but not used in 04A.
TEST_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": list(REFERENCE_CLASSES),
    "batch": [4],
}

# ---------------------------------------------------------------------
# Matrix search space
# ---------------------------------------------------------------------
RUN_ALL_PIXELS_STANDARD = False
RUN_EMPIRICAL_FOR_ALL_PIXELS = False

STANDARD_MATRIX_METHODS = [
    "object_mean",
    "object_median",
    "balanced_pixels",
]

if RUN_ALL_PIXELS_STANDARD:
    STANDARD_MATRIX_METHODS.append("all_pixels")

# ---------------------------------------------------------------------
# SIMCA rules
# ---------------------------------------------------------------------
SIMCA_RULE_VARIANTS_MAIN = (
    "simple_chi2",
    "simple_emp_cv",
    "alternative_chi2_fixed2",
    "alternative_chi2_emp_cv",
    "alternative_empHQ_fixed2",
    "alternative_empHQ_emp_cv",
    "data_driven_chi2",
    "data_driven_emp_cv",
    "combined_index_chi2",
)

# ---------------------------------------------------------------------
# Preprocessing search space
# ---------------------------------------------------------------------
DEFAULT_PREPROCESSING_CONFIGS = {
    "snv": ("snv",),
    "absorbance": ("absorbance",),
    "absorbance_snv": ("absorbance", "snv"),
    "absorbance_sg_smooth": ("absorbance", "sg_smooth"),
    "absorbance_sg_d1": ("absorbance", "sg_d1"),
    "absorbance_snv_sg_smooth": ("absorbance", "snv", "sg_smooth"),
    "absorbance_snv_sg_d1": ("absorbance", "snv", "sg_d1"),
    "snv_sg_d1": ("snv", "sg_d1"),
}


def _parse_preprocessing_steps(value):
    if isinstance(value, (list, tuple)):
        return tuple(str(v) for v in value)

    value = str(value)

    if "+" in value:
        return tuple(v.strip() for v in value.split("+") if v.strip())

    return (value.strip(),)


if PCA_SELECTED_PREPROCESSINGS_PATH.exists():
    pca_selected_preprocessings_df = load_parquet(PCA_SELECTED_PREPROCESSINGS_PATH)

    PREPROCESSING_CONFIGS = {
        str(row["preprocessing"]): _parse_preprocessing_steps(row["preprocessing_steps"])
        for _, row in pca_selected_preprocessings_df.drop_duplicates("preprocessing").iterrows()
    }

    print("Loaded preprocessing shortlist from PCA notebook:")
    print(PCA_SELECTED_PREPROCESSINGS_PATH)

else:
    pca_selected_preprocessings_df = pd.DataFrame()
    PREPROCESSING_CONFIGS = DEFAULT_PREPROCESSING_CONFIGS.copy()

    print("[WARNING] PCA preprocessing shortlist not found.")
    print("Using default preprocessing search space instead:")
    print(PCA_SELECTED_PREPROCESSINGS_PATH)

PREPROCESSING_CONFIGS = normalize_preprocessing_configs(PREPROCESSING_CONFIGS)

# ---------------------------------------------------------------------
# Hyperparameter search space
# ---------------------------------------------------------------------
N_COMPONENTS_VALUES = [3, 4, 5, 6, 7, 8, 10, 11, 12]
ALPHA_VALUES = [0.05, 0.01]
OBJECT_THRESHOLDS = [0.70, 0.75, 0.80, 0.85, 0.90]

M_VALUES = [40]
BALANCED_PIXEL_STRATEGY_VALUES = ["random", "center"]

SG_WINDOW_LENGTH_VALUES = [11]
SG_POLYORDER_VALUES = [2]
POSITION_DILATION_RADIUS_VALUES = [3]

DEFAULT_M = 40
DEFAULT_SG_WINDOW_LENGTH = 11
DEFAULT_SG_POLYORDER = 2

REPLACE_BALANCED_PIXELS = False
RANDOM_STATE = 42

CV_N_SPLITS = 5
CV_GROUP_COL = "object_id"

TWO_WAY_PARETO_PASSES = [
    {
        "name": "within_same_model_preprocessing",
        "group_cols": [
            "matrix_family",
            "training_matrix_id",
            "rule_variant",
            "limit_source",
            "n_components",
            "alpha",
            "object_threshold",
            "sg_window_length",
            "sg_polyorder",
            "position_dilation_radius",
            "m_effective",
            "balanced_pixel_strategy_effective",
        ],
        "minimize_cols": ["fn_rate", "fp_rate"],
        "maximize_cols": ["balanced_accuracy"],
    },
    {
        "name": "within_same_family_components",
        "group_cols": [
            "matrix_family",
            "n_components",
            "object_threshold",
        ],
        "minimize_cols": ["fn_rate", "fp_rate"],
        "maximize_cols": ["balanced_accuracy"],
    },
    {
        "name": "family_level_pareto",
        "group_cols": [
            "matrix_family",
        ],
        "minimize_cols": ["fn_rate", "fp_rate"],
        "maximize_cols": ["balanced_accuracy"],
    },
]

# ---------------------------------------------------------------------
# Runtime flags
# ---------------------------------------------------------------------
RUN_STANDARD_GRID = False
RUN_EMPIRICAL_CV_GRID = True
RUN_DIAGNOSTIC_PLOTS = True

# Important: broad panel, not final models.
N_SELECTED_PER_TRAINING_MATRIX = 8
N_SELECTED_PER_MATRIX_FAMILY = 25
N_SELECTED_OVERALL = 50

print("DB_H5_PATH:", DB_H5_PATH)
print("RESULTS_DIR:", RESULTS_DIR)
print("WAVELENGTH_MODE:", WAVELENGTH_MODE)
print("RESULTS_TAG:", RESULTS_TAG)
print("TRAIN_FILTERS:", TRAIN_FILTERS)
print("VALIDATION_FILTERS:", VALIDATION_FILTERS)
print("STANDARD_MATRIX_METHODS:", STANDARD_MATRIX_METHODS)
print("Number of preprocessing configs:", len(PREPROCESSING_CONFIGS))

Loaded preprocessing shortlist from PCA notebook:
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03_pca_non_noisy_all\pca_selected_preprocessings.parquet
DB_H5_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database.h5
RESULTS_DIR: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_validation_non_noisy_all
WAVELENGTH_MODE: non_noisy_all
RESULTS_TAG: non_noisy_all
TRAIN_FILTERS: {'sample_kind': ['pure'], 'object_nut_type': ['peanut'], 'batch': [1, 2]}
VALIDATION_FILTERS: {'sample_kind': ['pure'], 'object_nut_type': ['almond', 'peanut'], 'batch': [3]}
STANDARD_MATRIX_METHODS: ['object_mean', 'object_median', 'balanced_pixels']
Number of preprocessing configs: 13


In [4]:
if not DB_H5_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_H5_PATH}. Run notebook 00 first.")

object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=WINDOW_MIN_NM,
        max_nm=WINDOW_MAX_NM,
    )

    wavelength_selection_df = wavelength_selection_summary(wavelength_info)

else:
    first_obj = next(iter(object_db.values()))
    wavelengths = first_obj.get("wavelengths")
    wavelengths = np.asarray(wavelengths, dtype=float) if wavelengths is not None else None
    wavelength_selection_df = pd.DataFrame()

if wavelengths is None:
    raise RuntimeError("No wavelength axis found in object_db.")

wavelength_config_df = pd.DataFrame([{
    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_bands": int(len(wavelengths)),
    "min_wavelength_nm": float(np.min(wavelengths)),
    "max_wavelength_nm": float(np.max(wavelengths)),
}])

print("Number of images:", len(image_db))
print("Number of objects:", len(object_db))
display(wavelength_config_df)

object_meta_df = pd.DataFrame([
    {
        "object_id": object_id,
        "source_image": obj.get("source_clean_key"),
        "sample_kind": obj.get("sample_kind"),
        "object_nut_type": obj.get("object_nut_type"),
        "batch": obj.get("batch"),
        "split": obj.get("split"),
        "area_pixels": obj.get("area_pixels"),
        "n_pixels": obj.get("n_pixels"),
        "n_bands": obj.get("n_bands"),
    }
    for object_id, obj in object_db.items()
])

display(
    object_meta_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .size()
    .reset_index(name="n_objects")
    .sort_values(["sample_kind", "object_nut_type", "batch"], na_position="last")
)

Number of images: 48
Number of objects: 1262


,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_bands,min_wavelength_nm,max_wavelength_nm
0,non_noisy_all,False,non_noisy_all,NaN,NaN,63,960.735294,1702.0


,sample_kind,object_nut_type,batch,n_objects
0,mixture,unknown,NaN,722
1,position_reference,peanut,1.0,47
2,position_reference,peanut,2.0,47
3,position_reference,peanut,3.0,47
4,position_reference,peanut,4.0,5
5,pure,almond,1.0,52
6,pure,almond,2.0,59
7,pure,almond,3.0,55
8,pure,almond,4.0,48
9,pure,peanut,1.0,46


In [5]:
def load_parquet_or_empty(path: Path) -> pd.DataFrame:
    """Load a parquet file if it exists, otherwise return an empty dataframe."""
    if Path(path).exists():
        return load_parquet(path)
    return pd.DataFrame()


def display_available_columns(df: pd.DataFrame, columns: list[str], n: int = 20):
    """Display only columns available in a dataframe."""
    available = [col for col in columns if col in df.columns]
    if not available:
        display(df.head(n))
    else:
        display(df[available].head(n))

# 2. Grid search

In [6]:
grid_summary_df, grid_results, grid_errors_df = run_simca_rule_variant_grid(
    object_db=object_db,
    image_db=image_db,
    train_filters=TRAIN_FILTERS,
    projection_filters=VALIDATION_FILTERS,
    preprocessing_configs=PREPROCESSING_CONFIGS,
    matrix_methods=["object_mean", "object_median", "balanced_pixels"],
    rule_variants=SIMCA_RULE_VARIANTS_MAIN,
    n_components_values=N_COMPONENTS_VALUES,
    alpha_values=ALPHA_VALUES,
    object_thresholds=OBJECT_THRESHOLDS,
    m_values=M_VALUES,
    balanced_pixel_strategy_values=BALANCED_PIXEL_STRATEGY_VALUES,
    sg_window_length_values=SG_WINDOW_LENGTH_VALUES,
    sg_polyorder_values=SG_POLYORDER_VALUES,
    position_dilation_radius_values=POSITION_DILATION_RADIUS_VALUES,
    random_state=42,
    wavelengths=wavelengths,
    keep_pixel_tables=False,
    keep_cv_tables=False,
    verbose=True,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
)


[1/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=3 | alpha=0.05 | SG=(11,2) | dilation=3

[2/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=3 | alpha=0.01 | SG=(11,2) | dilation=3

[3/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=4 | alpha=0.05 | SG=(11,2) | dilation=3

[4/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=4 | alpha=0.01 | SG=(11,2) | dilation=3

[5/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=5 | alpha=0.05 | SG=(11,2) | dilation=3

[6/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=5 | alpha=0.01 | SG=(11,2) | dilation=3

[7/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=6 | alpha=0.05 | SG=(11,2) | dilation=3

[8/936] empirical_cv | matrix=object_mean | preprocessing=absorbance_sg_d1 | A=6 | alpha=0.01 | SG=(11,2) | dilation=3

[9/936] empirical_cv | matrix=object_me

In [7]:
grid_summary_df = normalize_simca_rule_columns(grid_summary_df)
grid_summary_df["selection_split"] = "validation_batch_3"
grid_summary_df["selection_strategy"] = "04A_grid_rule_variant_universe"

save_parquet(grid_summary_df, GRID_SUMMARY_PATH)

display_cols = [
    "model_family",
    "matrix_family",
    "training_matrix_id",
    "matrix_method",
    "balanced_pixel_strategy",
    "balanced_pixel_strategy_effective",
    "m",
    "m_effective",
    "preprocessing",
    "rule",
    "rule_variant",
    "selected_rule_name",
    "rule_for_refit",
    "limit_source",
    "n_components",
    "alpha",
    "object_threshold",
    "sg_window_length",
    "sg_polyorder",
    "position_dilation_radius",
    "balanced_accuracy",
    "target_sensitivity",
    "non_target_specificity",
    "fn_rate",
    "fp_rate",
    "f1_score",
    "accuracy",
    "selection_score",
    "score_conservative_target",
    "score_balanced_reference",
    "score_specificity_control",
]
display_cols = [col for col in display_cols if col in grid_summary_df.columns]

display(grid_summary_df[display_cols].head(30))

,model_family,matrix_family,training_matrix_id,matrix_method,balanced_pixel_strategy,balanced_pixel_strategy_effective,m,m_effective,preprocessing,rule,rule_variant,selected_rule_name,rule_for_refit,limit_source,n_components,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,balanced_accuracy,target_sensitivity,non_target_specificity,fn_rate,fp_rate,f1_score,accuracy,selection_score
0,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_chi2,simple_chi2,simple_chi2,chi2,7,0.01,0.75,11,2,3,0.945455,1.0,0.890909,0.0,0.109091,0.946429,0.944444,-0.042881
1,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_chi2,simple_chi2,simple_chi2,chi2,7,0.01,0.70,11,2,3,0.936364,1.0,0.872727,0.0,0.127273,0.938053,0.935185,-0.061666
2,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,7,0.05,0.75,11,2,3,0.927273,1.0,0.854545,0.0,0.145455,0.929825,0.925926,-0.080445
3,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,7,0.05,0.70,11,2,3,0.918182,1.0,0.836364,0.0,0.163636,0.921739,0.916667,-0.099216
4,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,alternative,alternative_chi2_fixed2,alternative_chi2_fixed2,alternative_chi2_fixed2,chi2,7,0.05,0.75,11,2,3,0.918182,1.0,0.836364,0.0,0.163636,0.921739,0.916667,-0.099216
5,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_d1,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,6,0.05,0.75,11,2,3,0.909091,1.0,0.818182,0.0,0.181818,0.913793,0.907407,-0.117980
6,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_d1,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,6,0.05,0.75,11,2,3,0.909091,1.0,0.818182,0.0,0.181818,0.913793,0.907407,-0.117980
7,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,alternative,alternative_chi2_emp_cv,alternative_chi2_emp_cv,alternative_chi2_emp_cv,empirical_cv,7,0.05,0.75,11,2,3,0.909091,1.0,0.818182,0.0,0.181818,0.913793,0.907407,-0.117980
8,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,7,0.05,0.75,11,2,3,0.909091,1.0,0.818182,0.0,0.181818,0.913793,0.907407,-0.117980
9,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,alternative,alternative_chi2_fixed2,alternative_chi2_fixed2,alternative_chi2_fixed2,chi2,7,0.05,0.70,11,2,3,0.900000,1.0,0.800000,0.0,0.200000,0.905983,0.898148,-0.136738


In [8]:
print("Columns available in grid_summary_df:")
print(sorted(grid_summary_df.columns))

print("\nNumber of configurations by family:")
display(
    grid_summary_df
    .groupby(["model_family", "matrix_family", "training_matrix_id"], dropna=False)
    .size()
    .reset_index(name="n_configs")
    .sort_values(["model_family", "matrix_family", "training_matrix_id"])
)

print("\nTop validation configurations:")
display(grid_summary_df[display_cols].head(30))

if RUN_DIAGNOSTIC_PLOTS and len(grid_summary_df) > 0:
    plot_metric_heatmap(
        grid_summary_df,
        index_col="preprocessing",
        columns_col="selected_rule_name",
        value_col="balanced_accuracy",
        title="Validation balanced accuracy by preprocessing and SIMCA rule",
        show=True,
    )

    plot_metric_heatmap(
        grid_summary_df,
        index_col="preprocessing",
        columns_col="selected_rule_name",
        value_col="fn_rate",
        title="Validation FN rate by preprocessing and SIMCA rule",
        show=True,
    )
else:
    print("Diagnostic plots skipped.")

Columns available in grid_summary_df:
['H_emp_cv', 'Q_emp_cv', 'accuracy', 'alpha', 'alternative_chi2_emp_cv', 'alternative_empHQ_emp_cv', 'balanced_accuracy', 'balanced_pixel_strategy', 'balanced_pixel_strategy_effective', 'cv_abs_rejection_error', 'cv_expected_rejection_rate', 'cv_n_splits', 'cv_rule_limit', 'cv_target_acceptance_rate', 'cv_target_rejection_rate', 'data_driven_emp_cv', 'f1_score', 'fn', 'fn_rate', 'fp', 'fp_rate', 'limit_source', 'm', 'm_effective', 'matrix_family', 'matrix_method', 'model_family', 'n', 'n_components', 'n_cv_groups', 'n_cv_observations', 'non_target_class', 'non_target_label', 'non_target_specificity', 'object_threshold', 'position_dilation_radius', 'precision', 'preprocessing', 'preprocessing_steps', 'rule', 'rule_for_refit', 'rule_original', 'rule_token', 'rule_variant', 'rule_variant_original', 'search_method', 'selected_rule_name', 'selection_score', 'selection_split', 'selection_strategy', 'sg_polyorder', 'sg_window_length', 'simple_emp_cv', 'ta

,model_family,matrix_family,training_matrix_id,n_configs
0,empirical_cv_rule,object_matrix,object_mean,10530
1,empirical_cv_rule,object_matrix,object_median,10530
2,empirical_cv_rule,pixel_matrix,balanced_pixel_center_m40,10530
3,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,10530



Top validation configurations:


,model_family,matrix_family,training_matrix_id,matrix_method,balanced_pixel_strategy,balanced_pixel_strategy_effective,m,m_effective,preprocessing,rule,rule_variant,selected_rule_name,rule_for_refit,limit_source,n_components,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,balanced_accuracy,target_sensitivity,non_target_specificity,fn_rate,fp_rate,f1_score,accuracy,selection_score
0,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_chi2,simple_chi2,simple_chi2,chi2,7,0.01,0.75,11,2,3,0.945455,1.0,0.890909,0.0,0.109091,0.946429,0.944444,-0.042881
1,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_chi2,simple_chi2,simple_chi2,chi2,7,0.01,0.70,11,2,3,0.936364,1.0,0.872727,0.0,0.127273,0.938053,0.935185,-0.061666
2,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,7,0.05,0.75,11,2,3,0.927273,1.0,0.854545,0.0,0.145455,0.929825,0.925926,-0.080445
3,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,7,0.05,0.70,11,2,3,0.918182,1.0,0.836364,0.0,0.163636,0.921739,0.916667,-0.099216
4,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,alternative,alternative_chi2_fixed2,alternative_chi2_fixed2,alternative_chi2_fixed2,chi2,7,0.05,0.75,11,2,3,0.918182,1.0,0.836364,0.0,0.163636,0.921739,0.916667,-0.099216
5,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_d1,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,6,0.05,0.75,11,2,3,0.909091,1.0,0.818182,0.0,0.181818,0.913793,0.907407,-0.117980
6,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_d1,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,6,0.05,0.75,11,2,3,0.909091,1.0,0.818182,0.0,0.181818,0.913793,0.907407,-0.117980
7,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,alternative,alternative_chi2_emp_cv,alternative_chi2_emp_cv,alternative_chi2_emp_cv,empirical_cv,7,0.05,0.75,11,2,3,0.909091,1.0,0.818182,0.0,0.181818,0.913793,0.907407,-0.117980
8,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,7,0.05,0.75,11,2,3,0.909091,1.0,0.818182,0.0,0.181818,0.913793,0.907407,-0.117980
9,empirical_cv_rule,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,random,random,40.0,40,absorbance_sg_smooth,alternative,alternative_chi2_fixed2,alternative_chi2_fixed2,alternative_chi2_fixed2,chi2,7,0.05,0.70,11,2,3,0.900000,1.0,0.800000,0.0,0.200000,0.905983,0.898148,-0.136738


# 3. Models selection

## 2-way

In [9]:
pareto_2way_df, pareto_audit_df = sequential_pareto_filter(
    grid_summary_df,
    passes=TWO_WAY_PARETO_PASSES,
)

display(pareto_2way_df.head(30))
display(pareto_audit_df.head(30))

,target_class,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,object_threshold,selection_score,search_method,model_family,matrix_family,training_matrix_id,matrix_method,m,m_effective,balanced_pixel_strategy,balanced_pixel_strategy_effective,preprocessing,preprocessing_steps,rule_variant,rule,n_components,alpha,non_target_label,sg_window_length,sg_polyorder,position_dilation_radius,cv_n_splits,n_cv_observations,n_cv_groups,H_emp_cv,Q_emp_cv,simple_emp_cv,alternative_chi2_emp_cv,alternative_empHQ_emp_cv,data_driven_emp_cv,cv_target_rejection_rate,cv_target_acceptance_rate,cv_expected_rejection_rate,cv_abs_rejection_error,cv_rule_limit,rule_original,rule_variant_original,rule_token,selected_rule_name,rule_for_refit,limit_source,selection_split,selection_strategy,kept_after_within_same_model_preprocessing,kept_after_within_same_family_components,kept_after_family_level_pareto
0,peanut,non_target,108,52,1,43,12,0.981132,0.218182,0.599657,0.592593,0.547368,0.702703,0.018868,0.781818,0.70,-0.923510,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_median,object_median,NaN,40,not_applicable,random,absorbance_sg_d2,absorbance+sg_d2,data_driven_emp_cv,data_driven,3,0.01,non_target,11,2,3,5,98,98,17.640046,2.147866e-09,4.319395,4.536808,1.380628,384.212136,0.010204,0.989796,0.01,0.000204,384.212136,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,True,True,True
1,peanut,non_target,108,49,4,37,18,0.924528,0.327273,0.625901,0.620370,0.569767,0.705036,0.075472,0.672727,0.70,-1.379785,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_median,object_median,NaN,40,not_applicable,random,absorbance_sg_d2,absorbance+sg_d2,simple_emp_cv,simple,3,0.01,non_target,11,2,3,5,98,98,17.640046,2.147866e-09,4.319395,4.536808,1.380628,384.212136,0.010204,0.989796,0.01,0.000204,4.319395,simple_emp_cv,simple_emp_cv,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,True,True,True
2,peanut,non_target,108,50,3,41,14,0.943396,0.254545,0.598971,0.592593,0.549451,0.694444,0.056604,0.745455,0.75,-1.264918,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_median,object_median,NaN,40,not_applicable,random,absorbance_sg_d2,absorbance+sg_d2,data_driven_emp_cv,data_driven,3,0.01,non_target,11,2,3,5,98,98,17.640046,2.147866e-09,4.319395,4.536808,1.380628,384.212136,0.010204,0.989796,0.01,0.000204,384.212136,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,True,True,True
3,peanut,non_target,108,46,7,33,22,0.867925,0.400000,0.633962,0.629630,0.582278,0.696970,0.132075,0.600000,0.75,-1.873314,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_median,object_median,NaN,40,not_applicable,random,absorbance_sg_d2,absorbance+sg_d2,simple_emp_cv,simple,3,0.01,non_target,11,2,3,5,98,98,17.640046,2.147866e-09,4.319395,4.536808,1.380628,384.212136,0.010204,0.989796,0.01,0.000204,4.319395,simple_emp_cv,simple_emp_cv,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,True,True,True
4,peanut,non_target,108,25,28,0,55,0.471698,1.000000,0.735849,0.740741,1.000000,0.641026,0.528302,0.000000,0.70,-5.236153,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_median,object_median,NaN,40,not_applicable,random,snv_sg_smooth,snv+sg_smooth,data_driven_chi2,data_driven,4,0.01,non_target,11,2,3,5,98,98,18.895228,2.341019e-01,3.433846,4.875279,2.000000,76.558987,NaN,NaN,NaN,NaN,NaN,data_driven_chi2,data_driven_chi2,data_driven_chi2,data_driven_chi2,data_driven_chi2,chi2,validation_batch_3,04A_grid_rule_variant_universe,True,True,True
5,peanut,non_target,108,25,28,0,55,0.471698,1.000000,0.735849,0.740741,1.000000,0.641026,0.528302,0.00

,pass_name,n_before,n_after,n_removed,removed_rate
0,within_same_model_preprocessing,42120,19145,22975,0.545465
1,within_same_family_components,19145,1079,18066,0.943641
2,family_level_pareto,1079,22,1057,0.979611


In [10]:
selected_04A_2way_df = (
    pareto_2way_df
    .sort_values(
        ["matrix_family", "fn_rate", "fp_rate", "balanced_accuracy"],
        ascending=[True, True, True, False],
    )
    .groupby("matrix_family", group_keys=False)
    .head(60)
    .reset_index(drop=True)
)

selected_04A_2way_df["selected_config_id"] = [
    f"04A_{row.matrix_family}_{i:04d}"
    for i, row in enumerate(selected_04A_2way_df.itertuples(), start=1)
]

In [11]:
(
    refit_04A_metrics_df,
    refit_04A_objects_df,
    refit_04A_pixels_df,
    refit_04A_pixel_errors_df,
    refit_04A_errors_df,
) = refit_selected_simca_configs(
    selected_configs_df=selected_04A_2way_df,
    object_db=object_db,
    image_db=image_db,
    train_filters=TRAIN_FILTERS,
    projection_filters=VALIDATION_FILTERS,
    preprocessing_configs=PREPROCESSING_CONFIGS,
    evaluation_split="validation_batch_3_refit",
    wavelengths=wavelengths,
    random_state=42,
    cv_n_splits=5,
    cv_group_col="object_id",
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
)

[validation_batch_3_refit] 04A_object_matrix_0001
[validation_batch_3_refit] 04A_object_matrix_0002
[validation_batch_3_refit] 04A_object_matrix_0003
[validation_batch_3_refit] 04A_object_matrix_0004
[validation_batch_3_refit] 04A_object_matrix_0005
[validation_batch_3_refit] 04A_object_matrix_0006
[validation_batch_3_refit] 04A_object_matrix_0007
[validation_batch_3_refit] 04A_object_matrix_0008
[validation_batch_3_refit] 04A_object_matrix_0009
[validation_batch_3_refit] 04A_object_matrix_0010
[validation_batch_3_refit] 04A_object_matrix_0011
[validation_batch_3_refit] 04A_object_matrix_0012
[validation_batch_3_refit] 04A_object_matrix_0013
[validation_batch_3_refit] 04A_object_matrix_0014
[validation_batch_3_refit] 04A_object_matrix_0015
[validation_batch_3_refit] 04A_object_matrix_0016
[validation_batch_3_refit] 04A_pixel_matrix_0017
[validation_batch_3_refit] 04A_pixel_matrix_0018
[validation_batch_3_refit] 04A_pixel_matrix_0019
[validation_batch_3_refit] 04A_pixel_matrix_0020
[val

## 3-way

In [12]:
THREE_WAY_CONFIG_COLS = ["selected_config_id"]

three_way_grid_04A_df, three_way_selected_04A_df = calibrate_three_way_thresholds_by_config(
    object_df=refit_04A_objects_df,
    config_cols=THREE_WAY_CONFIG_COLS,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
    lower_thresholds=np.round(np.arange(0.05, 0.61, 0.05), 2),
    upper_thresholds=np.round(np.arange(0.40, 0.96, 0.05), 2),
    max_target_miss_rate=0.00,
    max_false_accept_rate=0.35,
    max_uncertain_rate=0.60,
)

In [13]:
selected_04A_candidates_df = selected_04A_2way_df.merge(
    three_way_selected_04A_df[
        [
            "selected_config_id",
            "three_way_lower_threshold",
            "three_way_upper_threshold",
            "target_miss_rate",
            "non_target_false_accept_rate",
            "uncertain_rate",
            "coverage_rate",
        ]
    ].rename(
        columns={
            "target_miss_rate": "val3_target_miss_rate",
            "non_target_false_accept_rate": "val3_false_accept_rate",
            "uncertain_rate": "val3_uncertain_rate",
            "coverage_rate": "val3_coverage_rate",
        }
    ),
    on="selected_config_id",
    how="left",
)

display(selected_04A_candidates_df.head(30))

save_parquet(grid_summary_df, GRID_SUMMARY_PATH)
save_parquet(pareto_2way_df, PARETO_2WAY_PATH)
save_parquet(three_way_grid_04A_df, THREE_WAY_GRID_PATH)
save_parquet(selected_04A_candidates_df, SELECTED_CANDIDATE_CONFIGS_PATH)

,target_class,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,object_threshold,selection_score,search_method,model_family,matrix_family,training_matrix_id,matrix_method,m,m_effective,balanced_pixel_strategy,balanced_pixel_strategy_effective,preprocessing,preprocessing_steps,rule_variant,rule,n_components,alpha,non_target_label,sg_window_length,sg_polyorder,position_dilation_radius,cv_n_splits,n_cv_observations,n_cv_groups,H_emp_cv,Q_emp_cv,simple_emp_cv,alternative_chi2_emp_cv,alternative_empHQ_emp_cv,data_driven_emp_cv,cv_target_rejection_rate,cv_target_acceptance_rate,cv_expected_rejection_rate,cv_abs_rejection_error,cv_rule_limit,rule_original,rule_variant_original,rule_token,selected_rule_name,rule_for_refit,limit_source,selection_split,selection_strategy,kept_after_within_same_model_preprocessing,kept_after_within_same_family_components,kept_after_family_level_pareto,selected_config_id,three_way_lower_threshold,three_way_upper_threshold,val3_target_miss_rate,val3_false_accept_rate,val3_uncertain_rate,val3_coverage_rate
0,peanut,non_target,108,53,0,52,3,1.000000,0.054545,0.527273,0.518519,0.504762,0.670886,0.000000,0.945455,0.75,-0.901540,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_mean,object_mean,NaN,40,not_applicable,random,absorbance_sg_smooth,absorbance+sg_smooth,data_driven_emp_cv,data_driven,11,0.01,non_target,11,2,3,5,98,98,66.668455,5.685526e-05,16.527747,17.928278,1.512749,1333.969396,0.010204,0.989796,0.01,0.000204,1333.969396,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,True,True,True,04A_object_matrix_0001,0.05,0.95,0.0,0.545455,0.435185,0.564815
1,peanut,non_target,108,52,1,43,12,0.981132,0.218182,0.599657,0.592593,0.547368,0.702703,0.018868,0.781818,0.70,-0.923510,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_median,object_median,NaN,40,not_applicable,random,absorbance_sg_d2,absorbance+sg_d2,data_driven_emp_cv,data_driven,3,0.01,non_target,11,2,3,5,98,98,17.640046,2.147866e-09,4.319395,4.536808,1.380628,384.212136,0.010204,0.989796,0.01,0.000204,384.212136,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,True,True,True,04A_object_matrix_0002,0.55,0.95,0.0,0.218182,0.712963,0.287037
2,peanut,non_target,108,50,3,41,14,0.943396,0.254545,0.598971,0.592593,0.549451,0.694444,0.056604,0.745455,0.75,-1.264918,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_median,object_median,NaN,40,not_applicable,random,absorbance_sg_d2,absorbance+sg_d2,data_driven_emp_cv,data_driven,3,0.01,non_target,11,2,3,5,98,98,17.640046,2.147866e-09,4.319395,4.536808,1.380628,384.212136,0.010204,0.989796,0.01,0.000204,384.212136,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,True,True,True,04A_object_matrix_0003,0.55,0.95,0.0,0.218182,0.712963,0.287037
3,peanut,non_target,108,49,4,37,18,0.924528,0.327273,0.625901,0.620370,0.569767,0.705036,0.075472,0.672727,0.70,-1.379785,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_median,object_median,NaN,40,not_applicable,random,absorbance_sg_d2,absorbance+sg_d2,simple_emp_cv,simple,3,0.01,non_target,11,2,3,5,98,98,17.640046,2.147866e-09,4.319395,4.536808,1.380628,384.212136,0.010204,0.989796,0.01,0.000204,4.319395,simple_emp_cv,simple_emp_cv,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,True,True,True,04A_object_matrix_0004,0.55,0.95,0.0,0.109091,0.814815,0.185185
4,peanut,non_target,108,46,7,33,22,0.867925,0.400000,0.633962,0.629630,0.582278,0.696970,0.132075,0.600000,0.75,-1.873314,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_median,object_median,

WindowsPath('C:/Users/alixg/OneDrive - Université Paris-Dauphine/hsi_nuts/results/04A_simca_grid_validation_non_noisy_all/selected_candidate_configs.parquet')

## Pixel matrices

### Best balanced accuracy

In [14]:
#combined_summary_df[(combined_summary_df['matrix_family']=='pixel_matrix') & (combined_summary_df['balanced_accuracy']>0.92)].sort_values('balanced_accuracy', ascending=False)[cols].drop_duplicates()

In [15]:
#selected_indices = combined_summary_df[(combined_summary_df['matrix_family']=='pixel_matrix') & (combined_summary_df['balanced_accuracy']>0.92)].sort_values('balanced_accuracy', ascending=False)[cols].drop_duplicates().index
#pixel_best_ba = combined_summary_df.filter(items=selected_indices, axis=0)

### Best fn_rate

In [16]:
#combined_summary_df[(combined_summary_df['matrix_family']=='pixel_matrix') & (combined_summary_df['fn']==0) & (combined_summary_df['fp_rate']<0.50)].sort_values('fp_rate')[cols].drop_duplicates().head(20)

In [17]:
#selected_indices = combined_summary_df[(combined_summary_df['matrix_family']=='pixel_matrix') & (combined_summary_df['fn']==0) & (combined_summary_df['fp_rate']<0.50)].sort_values('fp_rate')[cols].drop_duplicates().head(20).index
#pixel_best_fn = combined_summary_df.filter(items=selected_indices, axis=0)

### Best score

In [18]:
#combined_summary_df[(combined_summary_df['matrix_family']=='pixel_matrix') ].sort_values('selection_score', ascending=False)[cols].drop_duplicates().head(15)

In [19]:
#selected_indices = combined_summary_df[(combined_summary_df['matrix_family']=='pixel_matrix') ].sort_values('selection_score', ascending=False)[cols].drop_duplicates().head(15).index
#pixel_best_selection_score = combined_summary_df.filter(items=selected_indices, axis=0)

In [20]:
#pixel_best_ba.shape[0], pixel_best_fn.shape[0], pixel_best_selection_score.shape[0]

In [21]:
#best_pixel = pd.concat([pixel_best_ba, pixel_best_fn, pixel_best_selection_score], axis=0).drop_duplicates()
#best_pixel[cols]

In [22]:
#best_pixel.shape[0]

## Object matrices

In [23]:
#combined_summary_df[combined_summary_df['matrix_family']=='object_matrix'][cols]

### Best balanced accuracy

In [24]:
#combined_summary_df[(combined_summary_df['matrix_family']=='object_matrix') & (combined_summary_df['balanced_accuracy']>0.71)][cols].drop_duplicates().sort_values('balanced_accuracy', ascending=False)

In [25]:
# selected_indices = combined_summary_df[(combined_summary_df['matrix_family']=='object_matrix') & (combined_summary_df['balanced_accuracy']>0.71)][cols].drop_duplicates().sort_values('balanced_accuracy', ascending=False).index
# object_best_ba = combined_summary_df.filter(items=selected_indices, axis=0)
# object_best_ba.shape[0]

### Best fn_rate

In [26]:
# object_best_fn = combined_summary_df[(combined_summary_df['matrix_family']=='object_matrix') & (combined_summary_df['fn']==0)].sort_values('fp_rate')
# object_best_fn[cols]

### Best selection score

In [27]:
# object_best_score = combined_summary_df[(combined_summary_df['matrix_family']=='object_matrix') & (combined_summary_df['non_target_specificity'] > 0.2)].sort_values('selection_score', ascending=False).head(10)
# object_best_score[cols]

In [28]:
# object_best_index = pd.concat([object_best_ba, object_best_fn, object_best_score])[cols].drop_duplicates().index
# object_best = combined_summary_df.filter(items=object_best_index, axis=0).sort_values('balanced_accuracy', ascending=False)
# object_best[cols]

## Automatic selection

In [29]:
# selected_candidates_df = select_top_models(
#     combined_summary_df,
#     n_per_training_matrix=N_SELECTED_PER_TRAINING_MATRIX,
#     n_per_matrix_family=N_SELECTED_PER_MATRIX_FAMILY,
#     n_overall=N_SELECTED_OVERALL,
# )

# selected_candidates_df = ensure_candidate_columns(selected_candidates_df)
# selected_candidates_df = normalize_simca_rule_columns(selected_candidates_df)
# selected_candidates_df = fill_selected_config_defaults(
#     selected_candidates_df,
#     default_values={
#         "target_class": TARGET_CLASS,
#         "non_target_label": NON_TARGET_LABEL,
#         "m": DEFAULT_M,
#         "sg_window_length": DEFAULT_SG_WINDOW_LENGTH,
#         "sg_polyorder": DEFAULT_SG_POLYORDER,
#         "position_dilation_radius": POSITION_DILATION_RADIUS_VALUES[0],
#         "alpha": ALPHA_VALUES[0],
#         "object_threshold": OBJECT_THRESHOLDS[0],
#     },
# )

# selected_candidates_df = add_detection_selection_score(selected_candidates_df)
# selected_candidates_df = add_reference_selection_scores(selected_candidates_df)
# selected_candidates_df = sort_detection_selection(selected_candidates_df, add_score=False)

# selected_candidates_df = selected_candidates_df.reset_index(drop=True)
# selected_candidates_df["selected_config_id"] = [
#     f"valcand_{i:03d}" for i in range(len(selected_candidates_df))
# ]
# selected_candidates_df["selection_split"] = "validation_batch_3"
# selected_candidates_df["selection_strategy"] = "04A_grid_validation_broad_panel"

# keep_cols = [
#     "selected_config_id",
#     "selection_split",
#     "selection_strategy",

#     "model_family",
#     "matrix_family",
#     "training_matrix_id",
#     "matrix_method",
#     "balanced_pixel_strategy",
#     "balanced_pixel_strategy_effective",
#     "m",
#     "m_effective",

#     "preprocessing",
#     "preprocessing_steps",

#     "rule",
#     "rule_variant",
#     "selected_rule_name",
#     "rule_for_refit",
#     "limit_source",

#     "target_class",
#     "non_target_label",

#     "n",
#     "tp",
#     "fn",
#     "fp",
#     "tn",
#     "balanced_accuracy",
#     "target_sensitivity",
#     "non_target_specificity",
#     "fn_rate",
#     "fp_rate",
#     "f1_score",
#     "accuracy",
#     "precision",

#     "selection_score",
#     "score_conservative_target",
#     "score_balanced_reference",
#     "score_specificity_control",

#     "n_components",
#     "alpha",
#     "object_threshold",
#     "sg_window_length",
#     "sg_polyorder",
#     "position_dilation_radius",

#     "n_train_observations",
#     "n_projected_pixels",

#     "cv_target_rejection_rate",
#     "cv_target_acceptance_rate",
#     "cv_expected_rejection_rate",
#     "cv_abs_rejection_error",
#     "cv_rule_limit",
# ]

# keep_cols = [col for col in keep_cols if col in selected_candidates_df.columns]
# selected_candidates_df = selected_candidates_df[keep_cols].copy()

# if selected_candidates_df.empty:
#     raise RuntimeError("No candidate configuration was selected.")

# save_parquet(selected_candidates_df, SELECTED_CANDIDATE_CONFIGS_PATH)

# print("Selected candidate configurations:", selected_candidates_df.shape)
# print("Saved:", SELECTED_CANDIDATE_CONFIGS_PATH)

# display(selected_candidates_df)

## Selected models

In [30]:
parameter_tendencies_df = summarize_parameter_tendencies(
    grid_summary_df,
    top_fraction=0.15,
    min_top_n=20,
)

print("Parameter tendencies among top-ranked validation models:")
display(parameter_tendencies_df.head(80))

Parameter tendencies among top-ranked validation models:


,parameter,value,count,rate_in_top_models,matrix_family,n_top_models
0,alpha,0.01,2503,0.792339,object_matrix,3159
1,alpha,0.05,656,0.207661,object_matrix,3159
2,balanced_pixel_strategy,not_applicable,3159,1.000000,object_matrix,3159
3,m,NaN,3159,1.000000,object_matrix,3159
4,matrix_method,object_median,3011,0.953150,object_matrix,3159
5,matrix_method,object_mean,148,0.046850,object_matrix,3159
6,n_components,3,455,0.144033,object_matrix,3159
7,n_components,4,385,0.121874,object_matrix,3159
8,n_components,5,366,0.115859,object_matrix,3159
9,n_components,6,356,0.112694,object_matrix,3159


In [31]:
grid_validation_protocol_df = pd.DataFrame([{
    "db_h5_path": str(DB_H5_PATH),
    "results_dir": str(RESULTS_DIR),

    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_active_bands": int(len(wavelengths)) if wavelengths is not None else np.nan,

    "target_class": TARGET_CLASS,
    "non_target_label": NON_TARGET_LABEL,
    "reference_classes_json": json.dumps(list(REFERENCE_CLASSES)),

    "train_filters_json": json.dumps(TRAIN_FILTERS, default=str),
    "validation_filters_json": json.dumps(VALIDATION_FILTERS, default=str),
    "test_filters_json": json.dumps(TEST_FILTERS, default=str),

    "standard_matrix_methods_json": json.dumps(STANDARD_MATRIX_METHODS),
    "run_all_pixels_standard": bool(RUN_ALL_PIXELS_STANDARD),
    "run_empirical_for_all_pixels": bool(RUN_EMPIRICAL_FOR_ALL_PIXELS),

    "rule_names_json": json.dumps(SIMCA_RULE_VARIANTS_MAIN),

    "preprocessing_configs_json": json.dumps(
        {name: list(steps) for name, steps in PREPROCESSING_CONFIGS.items()},
        default=str,
    ),
    "pca_selected_preprocessings_path": str(PCA_SELECTED_PREPROCESSINGS_PATH),
    "used_pca_preprocessing_shortlist": bool(PCA_SELECTED_PREPROCESSINGS_PATH.exists()),

    "n_components_values_json": json.dumps(N_COMPONENTS_VALUES),
    "alpha_values_json": json.dumps(ALPHA_VALUES),
    "object_thresholds_json": json.dumps(OBJECT_THRESHOLDS),
    "m_values_json": json.dumps(M_VALUES),
    "balanced_pixel_strategy_values_json": json.dumps(BALANCED_PIXEL_STRATEGY_VALUES),
    "sg_window_length_values_json": json.dumps(SG_WINDOW_LENGTH_VALUES),
    "sg_polyorder_values_json": json.dumps(SG_POLYORDER_VALUES),
    "position_dilation_radius_values_json": json.dumps(POSITION_DILATION_RADIUS_VALUES),

    "random_state": int(RANDOM_STATE),
    "replace_balanced_pixels": bool(REPLACE_BALANCED_PIXELS),
    "cv_n_splits": int(CV_N_SPLITS) if CV_N_SPLITS is not None else np.nan,
    "cv_group_col": CV_GROUP_COL,

    "n_grid_rows": int(len(grid_summary_df)),
    "n_selected_candidate_configs": int(len(selected_04A_candidates_df)),

    "grid_summary_path": str(SIMCA_RULE_VARIANTS_MAIN),
    "combined_grid_summary_path": str(GRID_SUMMARY_PATH),
    "selected_candidate_configs_path": str(SELECTED_CANDIDATE_CONFIGS_PATH),
}])

save_parquet(grid_validation_protocol_df, GRID_VALIDATION_PROTOCOL_PATH)

print("Saved grid validation protocol:")
print(GRID_VALIDATION_PROTOCOL_PATH)

display(grid_validation_protocol_df)

Saved grid validation protocol:
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_validation_non_noisy_all\grid_validation_protocol.parquet


,db_h5_path,results_dir,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_active_bands,target_class,non_target_label,reference_classes_json,train_filters_json,validation_filters_json,test_filters_json,standard_matrix_methods_json,run_all_pixels_standard,run_empirical_for_all_pixels,rule_names_json,preprocessing_configs_json,pca_selected_preprocessings_path,used_pca_preprocessing_shortlist,n_components_values_json,alpha_values_json,object_thresholds_json,m_values_json,balanced_pixel_strategy_values_json,sg_window_length_values_json,sg_polyorder_values_json,position_dilation_radius_values_json,random_state,replace_balanced_pixels,cv_n_splits,cv_group_col,n_grid_rows,n_selected_candidate_configs,grid_summary_path,combined_grid_summary_path,selected_candidate_configs_path
0,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,non_noisy_all,False,non_noisy_all,NaN,NaN,63,peanut,non_target,"[""almond"", ""peanut""]","{""sample_kind"": [""pure""], ""object_nut_type"": [...","{""sample_kind"": [""pure""], ""object_nut_type"": [...","{""sample_kind"": [""pure""], ""object_nut_type"": [...","[""object_mean"", ""object_median"", ""balanced_pix...",False,False,"[""simple_chi2"", ""simple_emp_cv"", ""alternative_...","{""absorbance_sg_d1"": [""absorbance"", ""sg_d1""], ...",C:\Users\alixg\OneDrive - Université Paris-Dau...,True,"[3, 4, 5, 6, 7, 8, 10, 11, 12]","[0.05, 0.01]","[0.7, 0.75, 0.8, 0.85, 0.9]",[40],"[""random"", ""center""]",[11],[2],[3],42,False,5,object_id,42120,22,"('simple_chi2', 'simple_emp_cv', 'alternative_...",C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...


In [32]:
print("04A_simca_grid_validation.ipynb completed.")
print()
print("Essential outputs:")
print(" -", GRID_SUMMARY_PATH)
print(" -", GRID_SUMMARY_PATH)
print(" -", SELECTED_CANDIDATE_CONFIGS_PATH)
print(" -", GRID_VALIDATION_PROTOCOL_PATH)

print()
print("Summary:")
print(f" - Wavelength mode: {WAVELENGTH_MODE}")
print(f" - Active bands: {len(wavelengths) if wavelengths is not None else 'unknown'}")
print(f" - Grid rows: {len(grid_summary_df)}")
print(f" - Selected candidate configs: {len(selected_04A_candidates_df)}")
print()
print("Next notebook:")
print("04B_simca_validation_robustness.ipynb")

04A_simca_grid_validation.ipynb completed.

Essential outputs:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_validation_non_noisy_all\grid_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_validation_non_noisy_all\grid_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_validation_non_noisy_all\selected_candidate_configs.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_validation_non_noisy_all\grid_validation_protocol.parquet

Summary:
 - Wavelength mode: non_noisy_all
 - Active bands: 63
 - Grid rows: 42120
 - Selected candidate configs: 22

Next notebook:
04B_simca_validation_robustness.ipynb
